<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/06_secuencias/60_rnn_lstm_gru.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# RNN, LSTM y GRU

**Pregunta guía:** ¿Cómo conserva una red información temporal?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


**Requiere TensorFlow.** Una RNN actualiza
$h_t=\phi(W_xx_t+W_hh_{t-1}+b)$. El gradiente multiplica repetidamente
Jacobianos y puede desaparecer o explotar. LSTM introduce celda y
compuertas; GRU combina compuertas con menos parámetros. Compararemos las
tres en predicción de señales amortiguadas.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEMILLA=42; keras.utils.set_random_seed(SEMILLA); rng=np.random.default_rng(SEMILLA)
n=1800; longitud=60
t=np.linspace(0,6,longitud+1)
X=[]; y=[]
for _ in range(n):
    w=rng.uniform(.8,2.5); gamma=rng.uniform(.02,.25); fase=rng.uniform(0,2*np.pi)
    señal=np.exp(-gamma*t)*np.sin(w*t+fase)+rng.normal(0,.02,len(t))
    X.append(señal[:-1,None]); y.append(señal[-1])
X=np.asarray(X,dtype="float32"); y=np.asarray(y,dtype="float32")
orden=rng.permutation(n); train,val,test=orden[:1200],orden[1200:1500],orden[1500:]


In [ ]:
capas={"SimpleRNN":layers.SimpleRNN,"LSTM":layers.LSTM,"GRU":layers.GRU}
filas=[]; modelos={}
for nombre,Capa in capas.items():
    keras.backend.clear_session(); keras.utils.set_random_seed(SEMILLA)
    modelo=keras.Sequential([layers.Input((longitud,1)),Capa(24),layers.Dense(1)])
    modelo.compile(optimizer=keras.optimizers.Adam(1e-3),loss="mse")
    hist=modelo.fit(X[train],y[train],validation_data=(X[val],y[val]),epochs=35,batch_size=64,verbose=0,
                    callbacks=[keras.callbacks.EarlyStopping(patience=5,restore_best_weights=True)])
    mse=modelo.evaluate(X[test],y[test],verbose=0)
    filas.append({"modelo":nombre,"parámetros":modelo.count_params(),"MSE_test":mse,"épocas":len(hist.history["loss"])})
    modelos[nombre]=modelo
display(pd.DataFrame(filas).set_index("modelo"))


In [ ]:
mejor=min(filas,key=lambda f:f["MSE_test"])["modelo"]
pred=modelos[mejor].predict(X[test],verbose=0).ravel()
plt.scatter(y[test],pred,alpha=.5); límites=[min(y[test].min(),pred.min()),max(y[test].max(),pred.max())]
plt.plot(límites,límites,"k--"); plt.xlabel("siguiente valor real"); plt.ylabel("predicción"); plt.title(mejor); plt.show()


**Ejercicios:** aumente longitud sin cambiar unidades; registre norma de
gradiente; compare contra persistencia $\hat x_{t+1}=x_t$; prediga varios
pasos de forma autoregresiva y observe acumulación de error.
